In [ ]:
import pandas as pd
import plotly.express as px
from pyproj import Transformer

# ---- Config ----
ORDER = ["poor", "bad", "moderate", "good", "high", "nan"]
COLORS = {
    "poor":     "#E41A1C",  # rojo fuerte
    "bad":      "#F44336",  # rojo
    "moderate": "#FDB462",  # naranja
    "good":     "#8DD3C7",  # verde claro
    "high":     "#1B9E77",  # verde oscuro
    "nan":      "#9E9E9E"   # gris
}

def lambert93_to_wgs84(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    """
    Convierte columnas X,Y en EPSG:2154 (Lambert-93 Francia) a lon/lat WGS84.
    Devuelve un DataFrame con columnas nuevas: 'lon', 'lat'.
    """
    transformer = Transformer.from_crs(2154, 4326, always_xy=True)
    lon, lat = transformer.transform(df[x_col].to_numpy(), df[y_col].to_numpy())
    out = df.copy()
    out["lon"] = lon
    out["lat"] = lat
    return out

def plotly_france_l93(
    df: pd.DataFrame,
    x_col: str, y_col: str,
    class_col: str,        # ej. 'IBD_EQR_Status'
    hover_cols: list[str] | None = None,
    point_size: int = 9,
    zoom_start: int = 5
):
    # 1) conversión a WGS84
    df2 = lambert93_to_wgs84(df, x_col, y_col)

    # 2) normalizar clases y agregar 'nan' para faltantes
    classes = (
        df2[class_col]
        .astype(str).str.strip().str.lower()
    )
    # donde el valor original era NaN, rehacemos a 'nan'
    classes = classes.mask(df2[class_col].isna(), "nan")
    df2["_class_norm"] = pd.Categorical(classes, categories=ORDER, ordered=True)

    # 3) columnas de hover
    hover = hover_cols or []
    base_hover = ["lat", "lon"]
    hover_data = {c: True for c in (hover + base_hover)}

    # 4) plotly
    fig = px.scatter_mapbox(
        df2,
        lat="lat", lon="lon",
        color="_class_norm",
        category_orders={"_class_norm": ORDER},
        color_discrete_map=COLORS,
        hover_data=hover_data,
        zoom=zoom_start,
        height=650
    )
    fig.update_layout(
        mapbox_style="open-street-map",
        legend_title_text=class_col,
        margin=dict(l=0, r=0, t=30, b=0)
    )
    # tamaño del marcador y borde blanco
    fig.update_traces(marker=dict(size=point_size, opacity=0.95, line=dict(width=0.8, color="white")))

    return fig


In [2]:
codes = pd.read_parquet('data/processed/dep_codes.parquet')
train = pd.read_parquet('data/processed/clean_train.parquet')

In [5]:
codes

,SamplingOperations_code,CodeSite_SamplingOperations,Longitude_Lambert93,Latitude_Lambert93,Watershed,CodeDepartement,HERlvl1Code,HERlvl1Name,HERlvl2Code,HERlvl2Name,Altitude,Streamsize
0,S02000008_20170703,S02000008,1039055.0,6721258.0,Rhin-Meuse,68,18,ALSACE,73,Collines du Sundgau,0.0,TP
1,S02000008_20200708,S02000008,1039055.0,6721258.0,Rhin-Meuse,68,18,ALSACE,73,Collines du Sundgau,0.0,TP
2,S02000010_20070906,S02000010,1039381.0,6737723.0,Rhin-Meuse,68,18,ALSACE,62,Alsace- plaine,246.0,None
3,S02000010_20080811,S02000010,1039381.0,6737723.0,Rhin-Meuse,68,18,ALSACE,62,Alsace- plaine,246.0,None
4,S02000010_20090721,S02000010,1039381.0,6737723.0,Rhin-Meuse,68,18,ALSACE,62,Alsace- plaine,246.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...
49226,S06710040_20090617,S06710040,946543.0,6229144.0,Rhône-Méditerranée,83,6,MEDITERRANEEN,108,Maures Esterel,24.0,TP
49227,S06330110_20220913,S06330110,935367.0,6486679.0,Rhône-Méditerranée,38,5,JURA-PREALPES DU NORD,79,Massifs Calcaires Chartreuse Aravis,256.0,P
49228,S06330240_20220920,S06330240,933597.0,6478724.0,Rhône-Méditerranée,38,5,JURA-PREALPES DU NORD,79,Massifs Calcaires Chartreuse Aravis,243.0,TP
49229,S06830110_20220831,S06830110,931109.0,6472568.0,Rhône-Méditerranée,38,5,JURA-PREALPES DU NORD,79,Massifs Calcaires Chartreuse Aravis,235.0,TP


In [4]:
df= pd.merge(codes,train, on = 'SamplingOperations_code', how = 'left')

In [7]:

# ==== Ejemplo de uso ====
fig = plotly_france_l93(
    df, x_col="Latitude_Lambert93", y_col="Longitude_Lambert93",
    class_col="IBD_EQR_Status",
    hover_cols=["SamplingOperations_code", "HERlvl1Name"],  # opcional
    point_size=10, zoom_start=6
 )
fig.show()


C:\Users\narro\AppData\Local\Temp\ipykernel_19572\3671903580.py:54: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



ValueError: Invalid property specified for object of type plotly.graph_objs.scattermapbox.Marker: 'line'

Did you mean "size"?

    Valid properties:
        allowoverlap
            Flag to draw all symbols, even if they overlap.
        angle
            Sets the marker orientation from true North, in degrees
            clockwise. When using the "auto" default, no rotation
            would be applied in perspective views which is
            different from using a zero angle.
        anglesrc
            Sets the source reference on Chart Studio Cloud for
            `angle`.
        autocolorscale
            Determines whether the colorscale is a default palette
            (`autocolorscale: true`) or the palette determined by
            `marker.colorscale`. Has an effect only if in
            `marker.color` is set to a numerical array. In case
            `colorscale` is unspecified or `autocolorscale` is
            true, the default palette will be chosen according to
      